# Your data → review labels → MediaPipe → 12 metrics → ONE classifier → live box/risk/alarm

**We are keeping MediaPipe and your 12 metrics.** This notebook uses your two supplied datasets (5,814 images). You label/separate them first, then run feature extraction and training. No MobileNet or second posture classifier is used by this workflow.

95% is a target to evaluate, not a promised training outcome. Posture risk is not a medical/SIDS probability. Existing metrics-only accuracy was 18.8%; no new trained accuracy is claimed here. [Full instructions](https://github.com/kg905/SeniorDesignProject/blob/main/docs/TRAIN_AND_SHOWCASE.md).

**First session: complete sections 1–5 (label and export). You may stop there and share the reviewed CSV so we can retrain next.** No GPU is required. No more ZIP uploads/splitting are needed.


## 1. Clone source and your already-uploaded images

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = Path('/content/SeniorDesignProject')
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/kg905/SeniorDesignProject.git', str(ROOT)],
                   env=dict(os.environ, GIT_LFS_SKIP_SMUDGE='1'), check=True)
os.chdir(ROOT)
print('Using your repository:', ROOT)


## 2. Install the matching MediaPipe environment
Use one cv2 distribution. If Colab requests a restart, restart the session after installation, then rerun the path cells. The pip removal below affects only this temporary Colab environment, not your Pi.

In [ ]:
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
%pip install -r /content/SeniorDesignProject/training/requirements.txt
%pip install --no-deps -e /content/SeniorDesignProject
!apt-get update -qq
!apt-get install -y -qq git-lfs libgles2 libegl1 libgl1 libportaudio2


## 3. Mount private Drive and prepare your review copy
Drive saves your reviews between sessions. This uses the original supplied dataset index, rebases its paths, and prioritizes conflicting groups/unlabeled images. It never automatically confirms a publisher/model label.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path('/content/SeniorDesignProject')
os.chdir(ROOT)
from google.colab import drive
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/CribWatch')
WORK.mkdir(parents=True, exist_ok=True)
from cribwatch.labeling import prepare_review, export_reviewed, resolve_reviewed_conflicts
from cribwatch.data import read_csv, write_csv, review_widget
REVIEW = WORK / 'reviewed-images.csv'
if not REVIEW.exists():
    print(prepare_review(ROOT / 'artifacts/dataset_audit_v2/images.csv', REVIEW, root=ROOT))
else:
    print('Resuming saved review; existing labels preserved')
print('CSV:', REVIEW)
print('Reviewed:', sum(r.get('reviewed') == '1' for r in read_csv(REVIEW)))


## 4. YOU label/separate the images (no widgets required)

Run the cell below. It shows **one photo at a time** using normal notebook output. In the input box below the photo, type **b** (back), **s** (side), **p** (belly/prone), **u** (unclear), **n** (no baby), or **q** (stop), then press **Enter**.

**Each label is saved immediately. There is no Save Batch button in this mode.** After 12 images, rerun the cell for the next batch. You can stop/close Colab and resume the same CSV later; an image displayed but not labeled remains pending. `no_baby` is for background/empty-crib fault examples, not a fourth posture classifier.

Leave unknown identities blank. The reviewer preserves existing baby/session IDs. Optional face annotation can be entered as `p visible`, `p turned_away`, `p occluded`, or `p unknown`; a visible face does not change belly posture to back. No model/publisher suggestion is shown by this reviewer. Unresolved source conflicts remain quarantined.

The older dropdown interface remains optional: `review_widget(REVIEW, batch_size=12, blind=False, show_suggestions=False)`. Use it only if widgets render in your browser. The default below needs no custom widget manager.


In [ ]:
from cribwatch.review_console import review_console
review_console(REVIEW, batch_size=12)


## 5. Export your confirmed labels
The optional helper below resolves a publisher conflict only when every image in its original-source group has been human-reviewed and agrees on one known posture. Incomplete/disagreeing groups stay quarantined. Export keeps human-reviewed, non-conflicting labels only.

In [ ]:
print(resolve_reviewed_conflicts(REVIEW))


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
stamp = datetime.now(ZoneInfo('America/New_York')).strftime('%Y%m%d-%H%M%S')
EXPORT = WORK / ('review-export-' + stamp)
# folders=True makes COPIES in class folders, never training/test folders.
print(export_reviewed(REVIEW, EXPORT, folders=True))
print('Reviewed labels:', EXPORT / 'reviewed.csv')
print('You can stop here and share this CSV; the original supplied images need not be uploaded again.')


## Stop here if you are only preparing labels

Share `reviewed.csv` through an authorized private route. It contains SHA256 matches to your original images. If you add new private images, those must also be available to whoever trains. Do not push private recordings or identity keys to the public repo.

Next sections are for **after labels are ready**. They require enough source groups/classes. For a final 95% claim, new permissioned Arducam recordings need actual baby/session IDs and independent reviewed test labels. The original scraped images lack those identities. Use a doll for camera/alarm bench setup; never position an infant unsafely for data collection.


## 6. Verify the exact MediaPipe assets
Retrieve only the two original task assets through Git LFS, not historical archives/other models. Verify their pinned official SHA256 values before inference.

In [ ]:
subprocess.run(['git', 'lfs', 'install', '--local'], check=True)
subprocess.run(['git', 'lfs', 'pull', '--include=models/*.task', '--exclude='], check=True)
import hashlib
from cribwatch.assets import ASSETS
for name in ['pose_landmarker_lite.task', 'face_landmarker.task']:
    assert hashlib.sha256((ROOT / 'models' / name).read_bytes()).hexdigest() == ASSETS[name][1]
print('Official pose/face assets verified')


## 7. Extract the 12 features from reviewed images

The existing exact schema/preprocessing is preserved. Pose failures remain recorded; they must not disappear from evaluation. If pose-quality coverage remains below 95%, all-image 95% is unattainable under the current gate, regardless of classifier quality. Improve framing/data/landmark extraction before promising that target.


In [ ]:
from cribwatch.data import extract_manifest
RUN = WORK / ('training-' + stamp)
RUN.mkdir(exist_ok=False)
FEATURES = RUN / 'features.csv'
extract_manifest(EXPORT / 'reviewed.csv', FEATURES, ROOT / 'models', workers=1)
feature_rows = read_csv(FEATURES)
posture_rows = [r for r in feature_rows if r['posture'] in ['back','side','prone']]
coverage = sum(r['pose_ok'] == '1' for r in posture_rows) / len(posture_rows)
print('Usable pose coverage:', coverage)
if coverage < .95:
    print('Pose coverage is currently below the level needed for 95% all-image accuracy.')


## 8. Train ONE posture classifier and evaluate grouped held-out data

`source_groups` is the initial exploratory mode when identities are unknown. `baby_ids` requires real IDs and reviewed labels; every baby/session/known source group stays together. Do not invent IDs. Use a fresh, reserved independent cohort for the final claim; this public dataset has already been inspected.

The single Random Forest uses median imputation and class balancing. Confidence selection uses validation only. Unknown predictions count as errors. Do not repeatedly tune against the final test result.


In [ ]:
from cribwatch.learning import train
EVALUATION_MODE = 'source_groups'  # change to baby_ids only for actual identified reviewed data
if EVALUATION_MODE not in ['source_groups', 'baby_ids']:
    raise ValueError('Unknown evaluation mode')
MODEL = RUN / 'model'
report = train(FEATURES, MODEL, exploratory=(EVALUATION_MODE == 'source_groups'))
print(json.dumps(report, indent=2))
if not report['research_95pct_point_target_met']:
    print('95% TARGET NOT ESTABLISHED. Show the actual accuracy/coverage and uncertainty.')


## 9. Inspect the same pipeline on photos

Only after evaluation, inspect examples including errors/unknowns. This uses MediaPipe + the one trained classifier, a visible-landmark pose box, and posture-based risk categories. A high score is not high medical risk or proof of high accuracy.


In [ ]:
examples = [r['image_path'] for r in read_csv(MODEL / 'test_predictions.csv')][:6]
ANNOTATED = RUN / 'annotated'
subprocess.run([sys.executable, '-m', 'cribwatch.showcase', 'image', '--model', str(MODEL / 'posture.joblib'),
                '--models', str(ROOT / 'models'), '--showcase', '--output', str(ANNOTATED), *examples], check=True)
from IPython.display import display
from PIL import Image
for image in sorted(ANNOTATED.glob('*.jpg')):
    display(Image.open(image))


## 10. Build/download the handoff ZIP
Requires an actually trained classifier and its matching evaluation report. Packages one classifier and actual pose/face assets; no fabricated V10 joblib ranges/cutoffs. No private images are bundled automatically.

In [ ]:
ZIP = RUN / 'SIDS_Pi5_Deployment.zip'
subprocess.run([sys.executable, str(ROOT / 'deploy/single-model/build_handoff.py'),
                '--model', str(MODEL), '--output', str(ZIP)], check=True)
from google.colab import files
files.download(str(ZIP))
print('Also saved in Drive:', ZIP)


## 11. Pi live showcase and risk-triggered alarm

Follow the README inside the ZIP and [Pi handoff](https://github.com/kg905/SeniorDesignProject/blob/main/docs/PI5_HANDOFF.md). Preserve sids_env; use a new Python 3.13 ARM64 environment with MediaPipe 1.1.1, not the incompatible 0.10.18 CPython 3.11 wheel. ARM64 imports/camera/GPIO performance still need physical testing.

On the Pi desktop/HDMI:

```bash
python monitor_pi5.py --model models --models models --picamera --display --showcase \
  --alarm-risk moderate --log demo-recordings/run-001.jsonl --record demo-recordings/run-001.mp4
```

Press s for an annotated scan/photo and JSON, q to stop. The box/risk refers to the exact classified frame. Moderate risk is side, high is prone; unknown is never lower risk. `--alarm-risk moderate` alarms on sustained side/prone; `high` on prone only. A fault remains audible regardless of that posture threshold.

Adopt the bundled TCP 5000 receiver on Pi 4, securely provision one matching shared key, and add `--alarm-host <Pi4-IP>` on Pi 5. Do not mix UDP 5005/legacy HTTP or assume an unseen receiver's schema matches. Verify sound, latching, acknowledgement/mute and camera/inference/Wi-Fi failure behavior.

Then assess continuous recordings: missed events, time to alarm, false alarms/hour, unknown time. Show the separate evaluation report alongside the live demonstration; a few successful scans do not prove 95% or SIDS prevention.
